# Embedding random-walk + kemiripan cosine
        **graph › link_prediction** · notebook `ppmi_embedding_cosine`

        **Kapan dipakai:** Pasangan yang berada di wilayah graf yang sama (komunitas) cenderung terhubung; menangkap lebih dari 2 langkah.

        **Jebakan umum:** Embedding dilatih hanya pada sisi yang diketahui; simpul terisolasi tidak terwakili.

**Data demo:** Cora (2708 makalah, 5429 sitasi, 1433 fitur kata, 7 topik). 10% sisi disembunyikan sebagai positif uji; pasangan non-sisi acak sejumlah sama sebagai negatif. Metrik: **AUC** dan **AP**.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Format data: `nodes.csv` (id, label, fitur...) dan `edges.csv` (source, target; arah diabaikan).

In [ ]:
GRAPH_DIR = "data_latihan/graph/cora"
MODEL_KEY = "ppmi_embedding_cosine"
SEED = 42
OUT = out_dir("graph", "link_prediction", MODEL_KEY); set_seed(SEED)
FRAC, SEEDS = 0.1, [0, 1, 2]

## 2. Muat graf dan ringkasan

In [ ]:
from superdb.graph import load_graph
g = load_graph(GRAPH_DIR)
deg = np.asarray(g.A.sum(1)).ravel(); E = int(g.A.nnz // 2)
same = float((g.y[g.A.nonzero()[0]] == g.y[g.A.nonzero()[1]]).mean())
print(f"{g.n} simpul, {E} sisi, derajat rata-rata {deg.mean():.2f} (maks {int(deg.max())}), simpul terisolasi: {int((deg == 0).sum())}, fitur: {None if g.X is None else g.X.shape[1]}")
print(f"homofili sisi (porsi sisi yang menghubungkan simpul sekelas): {same:.2f}  -> {'graf homofili: tetangga informatif' if same > .6 else 'graf heterofili: tetangga kurang informatif'}")

## 3. Metode
`link_scores(A_train, pairs)` mengembalikan skor 'kemungkinan terhubung' tiap pasangan; hanya melihat graf yang sudah disembunyikan sisi ujinya.

In [ ]:
import numpy as np
from superdb.graph import pair_heuristics, link_split, sample_negatives, ppmi_embedding

def link_scores(A_train, pairs):
    Z = ppmi_embedding(A_train, dim=64)
    return (Z[pairs[:, 0]] * Z[pairs[:, 1]]).sum(1)

## 4. Evaluasi

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
import time; rows = []
for s in SEEDS:
    A_tr, pos, neg = link_split(g.A, FRAC, seed=s); pairs = np.vstack([pos, neg]); y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    t0 = time.time(); sc = np.asarray(link_scores(A_tr, pairs), float)
    rows.append({"seed": s, "AUC": roc_auc_score(y, sc), "AP": average_precision_score(y, sc), "detik": time.time() - t0, "n_uji": len(pairs)})
res = pd.DataFrame(rows); display(res.round(4)); metrics = {"AUC_mean": float(res.AUC.mean()), "AUC_std": float(res.AUC.std()), "AP_mean": float(res.AP.mean())}
print(f"AUC = {res.AUC.mean():.3f} ± {res.AUC.std():.3f} (acak = 0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5)); ax.hist(sc[y == 1], bins=30, alpha=.6, label="sisi asli (tersembunyi)", density=True); ax.hist(sc[y == 0], bins=30, alpha=.6, label="non-sisi", density=True); ax.legend(); ax.set_title("distribusi skor")
plt.tight_layout(); plt.savefig(OUT / "scores.png", dpi=130); plt.show()
save_json({"metrics": metrics, "per_seed": rows}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan heuristik (cepat, tanpa pelatihan) dengan supervised; selisih biasanya kecil pada graf sitasi.
- Evaluasi lomba sering memakai ranking (mis. MRR/Hits@K) terhadap banyak kandidat negatif: ganti metrik di sel evaluasi.